# HARN evaluation notebook

Evaluates a trained **HARN** (Hierarchical Associative Resonance Network) checkpoint
for one explicit **asset** and **training seed**, on the corresponding processed
**test** split, using the same normalization, target conversion, and streaming-state
handling as `train_harn.py`.

This notebook targets the current generic multi-asset / multi-seed HARN project layout:

```text
checkpoints/harn/<ASSET>/<SEED>/{best_model.pt, last_model.pt, training_history.csv}
data/processed/<ASSET>/{train.pkl, val.pkl, test.pkl}
```

Evaluation is entirely determined by two explicit variables set in the configuration
cell below:

```python
ASSET = "EURUSD"
SEED = 42
```

Changing only `ASSET` and `SEED` and re-running the notebook top to bottom evaluates a
different experiment — no other code changes are required, and no asset, seed, or
checkpoint is ever auto-selected.

It:

1. locates the project root and imports `harn.py` / `train_harn.py` directly from `src/models/`,
2. loads `checkpoints/harn/<ASSET>/<SEED>/<CHECKPOINT_NAME>` and `data/processed/<ASSET>/test.pkl`,
3. validates that checkpoint metadata (when present), the selected `ASSET`/`SEED`, and the
   processed data all agree — and raises immediately if they don't,
4. reconstructs the exact `ModelConfig` from the checkpoint's own tensor shapes (robust
   even if the checkpoint used non-default hyperparameters),
5. restores weights and calibration statistics,
6. runs a single continuous, stateful streaming pass over the full test set using
   `train_harn.StreamingBatcher`, with correct event-driven update flags for every
   timeframe above the base level,
7. predicts bps deltas and reconstructed absolute prices for every timeframe in the
   asset's own hierarchy (read from the processed payload, never hardcoded),
8. computes MAE, MSE, RMSE, R², MAPE, and directional accuracy for each timeframe (both
   price space and delta space),
9. plots actual-vs-predicted prices, delta predictions, residuals, error distributions,
   and scatter plots — all labeled with the asset and seed being evaluated.

## Reproducibility contract

- `ASSET` and `SEED` are the **only** variables that determine what gets evaluated.
  There is no automatic asset discovery and no fallback to a differently-named
  checkpoint or dataset.
- Checkpoint and processed-data paths are constructed directly from `ASSET` / `SEED`
  (`checkpoints/harn/<ASSET>/<SEED>/<CHECKPOINT_NAME>`, `data/processed/<ASSET>/test.pkl`)
  and the notebook fails loudly if either path is missing.
- The timeframe hierarchy (e.g. `('m15', 'h1', 'h4')` for EURUSD, `('m5', 'm15')` for
  XAUUSD) is read entirely from the processed payload — nothing in the evaluation logic
  assumes a fixed set or count of timeframes.
- `HARNDataset`, `StreamingBatcher`, and `_prepare_targets` are imported directly from
  `train_harn.py`, not reimplemented, so evaluation cannot silently drift from training.
- Checkpoint/asset/seed/timeframe mismatches raise an error and stop the notebook —
  they are never silently downgraded to a warning.

In [ ]:
%matplotlib inline
from __future__ import annotations

import pickle
import sys
import warnings
from dataclasses import fields
from pathlib import Path
from typing import Any, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("default")

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

In [ ]:
# ---------------------------------------------------------------------------
# Evaluation configuration — the ONLY cell that should change between runs
# ---------------------------------------------------------------------------
ASSET = "XAUUSD"  # one of AVAILABLE_ASSETS
SEED = 42
    
# "best_model.pt" is the checkpoint saved at the best validation loss and is the
# right default for final evaluation. Set this to "last_model.pt" if you
# specifically want to evaluate the final training epoch instead.
CHECKPOINT_NAME = "last_model.pt"

# Number of parallel contiguous streams StreamingBatcher splits the (chronologically
# ordered) test set into. NUM_STREAMS = 1 -> a single fully chronological streaming
# pass with persistent memory carried across the entire test set.
NUM_STREAMS = 1

AVAILABLE_ASSETS = ("AAPL", "EURUSD", "USDCHF", "XAUUSD")
AVAILABLE_SEEDS = (42, 151, 359)

if ASSET not in AVAILABLE_ASSETS:
    raise ValueError(
        f"Unknown ASSET {ASSET!r}. Expected one of {AVAILABLE_ASSETS}. "
        "Evaluation never auto-selects an asset -- fix ASSET and re-run."
    )

if SEED not in AVAILABLE_SEEDS:
    raise ValueError(
        f"Unknown SEED {SEED!r}. Expected one of {AVAILABLE_SEEDS}. "
        "Evaluation never auto-selects a seed -- fix SEED and re-run."
    )

print(f"Evaluating ASSET={ASSET!r}, SEED={SEED}, CHECKPOINT_NAME={CHECKPOINT_NAME!r}")

In [ ]:
# ---------------------------------------------------------------------------
# Locate project root
# ---------------------------------------------------------------------------
def find_project_root(start: "Path | None" = None) -> Path:
    """Walk upward from `start` until the canonical HARN project layout is found."""
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        has_model = (
            (candidate / "src" / "models" / "harn.py").exists()
            and (candidate / "src" / "models" / "train_harn.py").exists()
        )
        has_data_root = (candidate / "data" / "processed").exists()
        has_ckpt_root = (candidate / "checkpoints" / "harn").exists()
        if has_model and has_data_root and has_ckpt_root:
            return candidate
    raise FileNotFoundError(
        "Could not locate the HARN project root (expected src/models/harn.py, "
        "src/models/train_harn.py, data/processed/, and checkpoints/harn/ under the "
        "same directory). Set PROJECT_ROOT manually below if the layout is custom."
    )


PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Importing train_harn (not just harn) gives direct access to HARNDataset,
# StreamingBatcher, and _prepare_targets -- the exact same code path used during
# training, so evaluation cannot silently drift from it.
try:
    from src.models.harn import HARN, ModelConfig
    from src.models import train_harn
except ImportError:
    # Fallback for a train_harn.py that performs bare sibling imports
    # (e.g. `from harn import HARN`) rather than package-relative ones -- that
    # style requires src/models itself to be importable as a plain path.
    MODEL_SRC_DIR = PROJECT_ROOT / "src" / "models"
    if str(MODEL_SRC_DIR) not in sys.path:
        sys.path.insert(0, str(MODEL_SRC_DIR))
    from harn import HARN, ModelConfig
    import train_harn

print(f"Project root : {PROJECT_ROOT}")

In [ ]:
# ---------------------------------------------------------------------------
# Resolve asset-specific paths
# ---------------------------------------------------------------------------
checkpoint_path = (
    PROJECT_ROOT / "checkpoints" / "harn" / ASSET / str(SEED) / CHECKPOINT_NAME
)
test_payload_path = PROJECT_ROOT / "data" / "processed" / ASSET / "test.pkl"

if not checkpoint_path.exists():
    raise FileNotFoundError(
        f"No checkpoint found for ASSET={ASSET!r}, SEED={SEED}.\n"
        f"Expected checkpoint at: {checkpoint_path}"
    )

if not test_payload_path.exists():
    raise FileNotFoundError(
        f"No processed test data found for ASSET={ASSET!r}.\n"
        f"Expected data at: {test_payload_path}"
    )

print(f"Checkpoint path   : {checkpoint_path.relative_to(PROJECT_ROOT)}")
print(f"Test payload path : {test_payload_path.relative_to(PROJECT_ROOT)}")

In [ ]:
# ---------------------------------------------------------------------------
# Load test payload
# ---------------------------------------------------------------------------
def load_pickle(path: Path) -> Any:
    with path.open("rb") as handle:
        return pickle.load(handle)


test_payload = load_pickle(test_payload_path)

required_payload_keys = {"features", "targets", "is_update", "timeframes", "statistics"}
missing_payload = sorted(required_payload_keys.difference(test_payload))
if missing_payload:
    raise KeyError(
        f"The processed test payload is missing required keys: {missing_payload}. "
        "Expected the generic train_harn.py payload format."
    )

# The processed payload is the single source of truth for the ordered timeframe
# hierarchy -- nothing downstream hardcodes a specific timeframe set or count.
timeframes = tuple(str(tf) for tf in test_payload["timeframes"])
TF_LABELS = {tf: tf.upper() for tf in timeframes}

for tf in timeframes:
    if tf not in test_payload["features"]:
        raise KeyError(f"Missing features['{tf}'] in test payload")
    if tf not in test_payload["targets"]:
        raise KeyError(f"Missing targets['{tf}'] in test payload")
    if tf not in test_payload["is_update"]:
        raise KeyError(f"Missing is_update['{tf}'] in test payload")

n_test_samples = len(test_payload["features"][timeframes[0]])

print(f"Timeframes    : {timeframes}")
print(f"Test samples  : {n_test_samples}")
print(
    "Window shapes : "
    + ", ".join(
        f"{tf}={tuple(np.asarray(test_payload['features'][tf]).shape[1:])}"
        for tf in timeframes
    )
)

In [ ]:
# ---------------------------------------------------------------------------
# Load checkpoint
# ---------------------------------------------------------------------------
def torch_load_compat(path: Path) -> Any:
    """Load a checkpoint across PyTorch versions (weights_only kwarg support)."""
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def extract_state_dict_and_metadata(checkpoint_obj: Any) -> tuple[dict, dict, bool]:
    """Return `(state_dict, metadata, is_wrapped)` for raw or wrapped checkpoints.

    - `last_model.pt` (written by train_harn.py) is a wrapped checkpoint dict
      containing 'model_state_dict' plus experiment metadata (e.g. asset, seed,
      timeframes, epoch, best_val_loss).
    - `best_model.pt` is a raw state_dict with no embedded metadata.
    """
    if isinstance(checkpoint_obj, dict) and "model_state_dict" in checkpoint_obj:
        state_dict = checkpoint_obj["model_state_dict"]
        metadata = {k: v for k, v in checkpoint_obj.items() if k != "model_state_dict"}
        return state_dict, metadata, True

    if isinstance(checkpoint_obj, dict) and checkpoint_obj and all(
        torch.is_tensor(v) for v in checkpoint_obj.values()
    ):
        return checkpoint_obj, {}, False

    raise TypeError(
        "Unsupported checkpoint format. Expected a raw state_dict or a dict "
        "containing 'model_state_dict'."
    )


checkpoint_obj_raw = torch_load_compat(checkpoint_path)
state_dict, checkpoint_meta, is_wrapped = extract_state_dict_and_metadata(checkpoint_obj_raw)

print(f"Checkpoint format : {'wrapped checkpoint' if is_wrapped else 'raw state_dict'}")
if "epoch" in checkpoint_meta:
    print(f"Saved epoch       : {checkpoint_meta['epoch']}")
if "best_val_loss" in checkpoint_meta:
    print(f"Best val loss     : {checkpoint_meta['best_val_loss']}")

In [ ]:
# ---------------------------------------------------------------------------
# Validate asset / seed / timeframe consistency
# ---------------------------------------------------------------------------
def validate_consistency(
    checkpoint_meta: dict[str, Any],
    asset: str,
    seed: int,
    payload_timeframes: tuple[str, ...],
) -> None:
    """Raise if checkpoint metadata contradicts the selected experiment.

    A raw `best_model.pt` carries no metadata, so there is nothing to cross-check
    beyond what `strict=True` weight loading and the model-config reconstruction
    already enforce. A wrapped `last_model.pt` embeds its own asset/seed/timeframes,
    which must agree with the ASSET/SEED the user selected and with the processed
    test data -- otherwise the model and the data belong to different experiments
    and any resulting metrics would be meaningless.
    """
    problems = []

    ckpt_asset = checkpoint_meta.get("asset", checkpoint_meta.get("symbol"))
    if ckpt_asset is not None and str(ckpt_asset) != asset:
        problems.append(f"checkpoint asset {ckpt_asset!r} != selected ASSET {asset!r}")

    ckpt_seed = checkpoint_meta.get("seed")
    if ckpt_seed is not None and int(ckpt_seed) != int(seed):
        problems.append(f"checkpoint seed {ckpt_seed!r} != selected SEED {seed!r}")

    ckpt_timeframes = checkpoint_meta.get("timeframes")
    if ckpt_timeframes is not None:
        ckpt_timeframes = tuple(str(tf) for tf in ckpt_timeframes)
        if ckpt_timeframes != payload_timeframes:
            problems.append(
                f"checkpoint timeframes {ckpt_timeframes} != processed test-data "
                f"timeframes {payload_timeframes}"
            )

    if problems:
        raise ValueError(
            "Checkpoint / ASSET / SEED / data consistency check failed for "
            f"ASSET={asset!r}, SEED={seed}:\n  - " + "\n  - ".join(problems)
        )

    print("Consistency check passed: checkpoint metadata agrees with ASSET, SEED, and timeframes.")


validate_consistency(checkpoint_meta, ASSET, SEED, timeframes)

In [ ]:
# ---------------------------------------------------------------------------
# Reconstruct ModelConfig from checkpoint tensor shapes + metadata
# ---------------------------------------------------------------------------
def infer_model_config_from_state_dict(
    state_dict: dict[str, torch.Tensor],
    timeframes: Sequence[str],
    checkpoint_metadata: dict[str, Any] | None = None,
) -> ModelConfig:
    """Reconstruct ModelConfig from checkpoint shapes + metadata.

    Uses the first level (`timeframes[0]`) to read representative per-level shape
    parameters from indexed module weights (e.g. encoders.0.*, cells.0.*, heads.0.*).
    Hyperparameters that are not recoverable from tensor shapes fall back to
    ModelConfig's own defaults unless the checkpoint metadata overrides them --
    this matters for `best_model.pt`, which is a raw state_dict with no metadata.
    """
    metadata = checkpoint_metadata or {}
    timeframes = tuple(str(tf) for tf in timeframes)
    default = ModelConfig(timeframes=timeframes)

    cfg = ModelConfig(
        timeframes=timeframes,
        dropout=float(metadata.get("dropout", default.dropout)),
        history_len=int(metadata.get("history_len", default.history_len)),
        close_index=int(metadata.get("close_index", default.close_index)),
    )

    cfg.input_dim = int(state_dict["encoders.0.input_proj.weight"].shape[1])
    cfg.d_model = int(state_dict["encoders.0.input_proj.weight"].shape[0])
    cfg.kernel_size = int(state_dict["encoders.0.branches.0.depthwise.weight"].shape[-1])

    branch_count = sum(
        1
        for key in state_dict
        if key.startswith("encoders.0.branches.") and key.endswith(".depthwise.weight")
    )
    dilations_meta = metadata.get("dilations")
    if dilations_meta is not None and len(dilations_meta) == branch_count:
        cfg.dilations = tuple(int(x) for x in dilations_meta)
    elif branch_count == len(default.dilations):
        cfg.dilations = default.dilations
    else:
        # Exact dilation values are not serialized in state_dict. Use a sensible
        # fallback with the correct branch count.
        cfg.dilations = tuple(2 ** i for i in range(branch_count))
        warnings.warn(
            "Dilation values are not recoverable from state_dict; using fallback "
            f"{cfg.dilations} inferred from branch count={branch_count}.",
            RuntimeWarning,
        )

    cfg.mem_key_dim = int(state_dict["cells.0.to_key.weight"].shape[0])
    cfg.state_dim = int(state_dict["cells.0.to_value.weight"].shape[0])
    cfg.coupling_rank = int(state_dict["resonance.q_proj.weight"].shape[0])
    cfg.head_hidden = int(state_dict["heads.0.net.0.weight"].shape[0])

    if "target_scale" in state_dict:
        cfg.target_scale = float(state_dict["target_scale"])
    elif "target_scale" in metadata:
        cfg.target_scale = float(metadata["target_scale"])

    return cfg


model_config = infer_model_config_from_state_dict(state_dict, timeframes, checkpoint_meta)

print("Reconstructed ModelConfig (dynamic hierarchy + checkpoint shapes):")
for f in fields(model_config):
    print(f"  {f.name:14s}: {getattr(model_config, f.name)}")

In [ ]:
# ---------------------------------------------------------------------------
# Build HARN and restore weights + calibration statistics
# ---------------------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = HARN(model_config).to(device)
model.load_state_dict(state_dict, strict=True)
model.eval()
model.set_feature_statistics(test_payload["statistics"])

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Device               : {device}")
print(f"Trainable parameters : {n_params:,}")

In [ ]:
# ---------------------------------------------------------------------------
# Evaluation configuration summary
# ---------------------------------------------------------------------------
summary_lines = [
    "=" * 60,
    "HARN EVALUATION",
    "=" * 60,
    f"Asset          : {ASSET}",
    f"Seed           : {SEED}",
    f"Checkpoint     : checkpoints/harn/{ASSET}/{SEED}/{CHECKPOINT_NAME}",
    f"Dataset        : data/processed/{ASSET}/test.pkl",
    f"Timeframes     : {timeframes}",
    f"Test samples   : {n_test_samples}",
    f"Streams        : {NUM_STREAMS}",
    f"Device         : {device}",
    "=" * 60,
]
print("\n".join(summary_lines))

## Stateful streaming inference

The cell below runs the model over the full test set exactly the way
`train_harn.py::evaluate()` does: `train_harn.StreamingBatcher` splits the
(chronologically ordered) test set into parallel contiguous streams,
`model.reset_states(...)` initializes persistent memory once, and each step calls
`model(windows, update_flags)` with the real per-timeframe update flags (`is_<tf>_update`
for every level above the base timeframe).

Targets are converted from normalized values into bps deltas and absolute prices with
`train_harn._prepare_targets(...)`, using the exact same argument list as training.

Everything here is driven by `timeframes`, dictionary-keyed per timeframe — there is no
per-timeframe branching and no assumption about how many timeframes an asset has.

In [ ]:
# ---------------------------------------------------------------------------
# Stateful streaming inference
# ---------------------------------------------------------------------------
test_dataset = train_harn.HARNDataset(test_payload, timeframes=timeframes)
batcher = train_harn.StreamingBatcher(test_dataset, NUM_STREAMS)
model.reset_states(NUM_STREAMS, device)

if len(batcher) * NUM_STREAMS < len(test_dataset):
    dropped = len(test_dataset) - NUM_STREAMS * len(batcher)
    print(
        f"Note: {dropped} trailing sample(s) are dropped so the test set divides "
        f"evenly into {NUM_STREAMS} stream(s) of length {len(batcher)}."
    )

raw: dict[str, dict[str, list[np.ndarray]]] = {
    tf: {"pred_delta": [], "true_delta": [], "pred_price": [], "true_price": [], "baseline_price": []}
    for tf in timeframes
}
update_flag_logs: dict[str, list[np.ndarray]] = {tf: [] for tf in timeframes[1:]}

with torch.no_grad():
    for batch in batcher:
        windows = {tf: batch[tf].to(device) for tf in timeframes}
        update_flags = {tf: batch[f"is_{tf}_update"].to(device) for tf in timeframes[1:]}

        # Identical target conversion to training:
        targets = train_harn._prepare_targets(
            batch,
            test_dataset.statistics,
            device,
            timeframes,
            model_config.close_index,
            model_config.target_scale,
        )

        # Exact generic forward interface used by train_harn.py
        output = model(windows, update_flags)

        for tf in timeframes:
            raw[tf]["pred_delta"].append(output[f"pred_delta_{tf}"].cpu().numpy().reshape(-1))
            raw[tf]["true_delta"].append(targets[f"delta_{tf}"].cpu().numpy().reshape(-1))
            raw[tf]["pred_price"].append(output[f"pred_{tf}"].cpu().numpy().reshape(-1))
            raw[tf]["true_price"].append(targets[f"absolute_{tf}"].cpu().numpy().reshape(-1))
            raw[tf]["baseline_price"].append(output[f"baseline_{tf}"].cpu().numpy().reshape(-1))

        for tf in timeframes[1:]:
            update_flag_logs[tf].append(update_flags[tf].cpu().numpy())

        model.detach_states()  # parity with training cadence

results = {tf: {key: np.concatenate(vals) for key, vals in raw[tf].items()} for tf in timeframes}
update_flag_arr = {
    tf: (np.concatenate(chunks) if chunks else np.array([], dtype=bool))
    for tf, chunks in update_flag_logs.items()
}

n_predictions = len(results[timeframes[0]]["pred_price"])
print(f"Collected {n_predictions} streaming predictions per timeframe on {device}.")
for tf in timeframes[1:]:
    n_updates = int(update_flag_arr[tf].sum())
    print(f"{TF_LABELS[tf]} bar closes in this pass: {n_updates} / {n_predictions}")

In [ ]:
# ---------------------------------------------------------------------------
# Prediction DataFrame — tidy long-format view, tagged with the experiment
# ---------------------------------------------------------------------------
frames = []
for tf in timeframes:
    r = results[tf]
    frames.append(pd.DataFrame({
        "asset": ASSET,
        "seed": SEED,
        "timeframe": TF_LABELS[tf],
        "step": np.arange(len(r["pred_price"])),
        "true_price": r["true_price"],
        "pred_price": r["pred_price"],
        "baseline_price": r["baseline_price"],
        "true_delta_bps": r["true_delta"],
        "pred_delta_bps": r["pred_delta"],
    }))
predictions_df = pd.concat(frames, ignore_index=True)
predictions_df.head(10)

## Metrics

Metrics are computed twice per timeframe:

- **on the reconstructed absolute price** (`pred_<tf>` vs. the true future close) — the
  economically meaningful quantity,
- **on the stationary bps delta** (`pred_delta_<tf>` vs. the true bps delta) — what the
  network is actually trained to minimize.

Directional accuracy is `sign(predicted delta) == sign(actual delta)`, i.e. "did the
model call the direction correctly" — identical whether computed from the delta arrays
or from `predicted price − baseline` vs. `actual price − baseline`.

Both tables carry `Asset` / `Seed` so results from different experiments are never
ambiguous once exported or concatenated.

In [ ]:
# ---------------------------------------------------------------------------
# Price-space metrics
# ---------------------------------------------------------------------------
def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    """MAE, MSE, RMSE, R2, and MAPE between two 1-D arrays."""
    residual = y_pred - y_true
    mse = float(np.mean(residual ** 2))
    rmse = float(np.sqrt(mse))
    mae = float(np.mean(np.abs(residual)))

    ss_res = float(np.sum(residual ** 2))
    ss_tot = float(np.sum((y_true - y_true.mean()) ** 2))
    r2 = float(1.0 - ss_res / ss_tot) if ss_tot > 0 else float("nan")

    eps = np.finfo(np.float64).eps
    mape = float(np.mean(np.abs(residual) / np.maximum(np.abs(y_true), eps)) * 100.0)

    return {"MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2, "MAPE (%)": mape}


def directional_accuracy(true_delta: np.ndarray, pred_delta: np.ndarray) -> float:
    """Share of steps where the predicted and actual price-change signs agree."""
    return float(np.mean(np.sign(true_delta) == np.sign(pred_delta)) * 100.0)


price_rows = []
for tf in timeframes:
    r = results[tf]
    price_metrics = regression_metrics(r["true_price"], r["pred_price"])
    price_metrics["Directional Accuracy (%)"] = directional_accuracy(r["true_delta"], r["pred_delta"])
    price_rows.append({"Asset": ASSET, "Seed": SEED, "Timeframe": TF_LABELS[tf], **price_metrics})

price_metrics_df = pd.DataFrame(price_rows).set_index(["Asset", "Seed", "Timeframe"])

fmt = {
    "MAE": "{:.6f}", "MSE": "{:.8f}", "RMSE": "{:.6f}",
    "R2": "{:.6f}", "MAPE (%)": "{:.3f}", "Directional Accuracy (%)": "{:.2f}",
}

print(f"Reconstructed absolute-price metrics — {ASSET}, seed {SEED}:")
display(price_metrics_df.style.format(fmt))

In [ ]:
# ---------------------------------------------------------------------------
# Delta-space metrics
# ---------------------------------------------------------------------------
delta_rows = []
for tf in timeframes:
    r = results[tf]
    delta_metrics = regression_metrics(r["true_delta"], r["pred_delta"])
    delta_metrics["Directional Accuracy (%)"] = directional_accuracy(r["true_delta"], r["pred_delta"])
    delta_rows.append({"Asset": ASSET, "Seed": SEED, "Timeframe": TF_LABELS[tf], **delta_metrics})

delta_metrics_df = pd.DataFrame(delta_rows).set_index(["Asset", "Seed", "Timeframe"])

print(f"Basis-point delta metrics — {ASSET}, seed {SEED}")
print("(MAPE is unstable here since true deltas can sit arbitrarily close to zero -- treat it as a rough signal):")
display(delta_metrics_df.style.format(fmt))

## Visualizations

All plots are indexed by streaming step, i.e. chronological order through the test set
(with `NUM_STREAMS = 1` this is simply the test-set row order). Every figure has one row
per timeframe in `timeframes`, and every title is tagged with the asset and seed being
evaluated so a saved/exported figure is never ambiguous about which experiment it came
from.

In [ ]:
# ---------------------------------------------------------------------------
# Actual vs. predicted price
# ---------------------------------------------------------------------------
def plot_actual_vs_predicted_price(results, tf_labels, timeframes, title):
    fig, axes = plt.subplots(len(timeframes), 1, figsize=(15, max(4, 3.2 * len(timeframes))), sharex=False)
    axes = np.atleast_1d(axes)
    for ax, tf in zip(axes, timeframes):
        r = results[tf]
        ax.plot(r["true_price"], label="Actual", linewidth=1.3)
        ax.plot(r["pred_price"], label="Predicted", linewidth=1.1, alpha=0.85)
        ax.set_title(f"{tf_labels[tf]} — actual vs. predicted price")
        ax.set_xlabel("Streaming step (chronological test index)")
        ax.set_ylabel("Price")
        ax.legend(loc="best")
        ax.grid(True, alpha=0.25)
    fig.suptitle(title, fontsize=15, y=1.02)
    fig.tight_layout()
    plt.show()


plot_actual_vs_predicted_price(
    results,
    TF_LABELS,
    timeframes,
    f"HARN — {ASSET} — Seed {SEED} — Reconstructed Absolute Price",
)

In [ ]:
# ---------------------------------------------------------------------------
# Delta predictions
# ---------------------------------------------------------------------------
def plot_delta_predictions(results, tf_labels, timeframes, title):
    fig, axes = plt.subplots(len(timeframes), 1, figsize=(15, max(4, 3.2 * len(timeframes))), sharex=False)
    axes = np.atleast_1d(axes)
    for ax, tf in zip(axes, timeframes):
        r = results[tf]
        ax.plot(r["true_delta"], label="Actual delta (bps)", linewidth=1.1)
        ax.plot(r["pred_delta"], label="Predicted delta (bps)", linewidth=1.0, alpha=0.8)
        ax.axhline(0.0, color="black", linewidth=0.8, linestyle="--")
        ax.set_title(f"{tf_labels[tf]} — actual vs. predicted delta (bps)")
        ax.set_xlabel("Streaming step (chronological test index)")
        ax.set_ylabel("Delta (basis points)")
        ax.legend(loc="best")
        ax.grid(True, alpha=0.25)
    fig.suptitle(title, fontsize=15, y=1.02)
    fig.tight_layout()
    plt.show()


plot_delta_predictions(
    results,
    TF_LABELS,
    timeframes,
    f"HARN — {ASSET} — Seed {SEED} — Stationary Basis-Point Delta",
)

In [ ]:
# ---------------------------------------------------------------------------
# Residual analysis
# ---------------------------------------------------------------------------
def plot_residuals_over_time(results, tf_labels, timeframes, title):
    fig, axes = plt.subplots(len(timeframes), 1, figsize=(15, max(4, 3.2 * len(timeframes))))
    axes = np.atleast_1d(axes)
    for ax, tf in zip(axes, timeframes):
        residual = results[tf]["pred_price"] - results[tf]["true_price"]
        ax.plot(residual, color="tab:red", linewidth=0.9)
        ax.axhline(0.0, color="black", linewidth=1.0, linestyle="--")
        ax.set_title(f"{tf_labels[tf]} — price residual (predicted − actual)")
        ax.set_xlabel("Streaming step (chronological test index)")
        ax.set_ylabel("Residual (price)")
        ax.grid(True, alpha=0.25)
    fig.suptitle(title, fontsize=15, y=1.02)
    fig.tight_layout()
    plt.show()


plot_residuals_over_time(
    results,
    TF_LABELS,
    timeframes,
    f"HARN — {ASSET} — Seed {SEED} — Price Residuals",
)

In [ ]:
# ---------------------------------------------------------------------------
# Error distributions
# ---------------------------------------------------------------------------
def plot_error_distributions(results, tf_labels, timeframes, title):
    fig, axes = plt.subplots(len(timeframes), 2, figsize=(15, max(4, 3.2 * len(timeframes))))
    if len(timeframes) == 1:
        axes = np.array([axes])
    for row, tf in enumerate(timeframes):
        price_residual = results[tf]["pred_price"] - results[tf]["true_price"]
        delta_residual = results[tf]["pred_delta"] - results[tf]["true_delta"]

        ax_price, ax_delta = axes[row, 0], axes[row, 1]

        ax_price.hist(price_residual, bins=50, color="tab:blue", alpha=0.75)
        ax_price.axvline(0.0, color="black", linewidth=1.0, linestyle="--")
        ax_price.set_title(f"{tf_labels[tf]} — price residual distribution")
        ax_price.set_xlabel("Residual (price)")
        ax_price.set_ylabel("Count")

        ax_delta.hist(delta_residual, bins=50, color="tab:orange", alpha=0.75)
        ax_delta.axvline(0.0, color="black", linewidth=1.0, linestyle="--")
        ax_delta.set_title(f"{tf_labels[tf]} — delta (bps) residual distribution")
        ax_delta.set_xlabel("Residual (bps)")
        ax_delta.set_ylabel("Count")

    fig.suptitle(title, fontsize=15, y=1.02)
    fig.tight_layout()
    plt.show()


plot_error_distributions(
    results,
    TF_LABELS,
    timeframes,
    f"HARN — {ASSET} — Seed {SEED} — Error Distributions",
)

In [ ]:
# ---------------------------------------------------------------------------
# Scatter plots
# ---------------------------------------------------------------------------
def plot_scatter(results, tf_labels, timeframes, title):
    fig, axes = plt.subplots(len(timeframes), 2, figsize=(15, max(4, 3.2 * len(timeframes))))
    if len(timeframes) == 1:
        axes = np.array([axes])
    for row, tf in enumerate(timeframes):
        r = results[tf]

        ax_price = axes[row, 0]
        ax_price.scatter(r["true_price"], r["pred_price"], s=8, alpha=0.4)
        lo = float(min(r["true_price"].min(), r["pred_price"].min()))
        hi = float(max(r["true_price"].max(), r["pred_price"].max()))
        ax_price.plot([lo, hi], [lo, hi], color="black", linestyle="--", linewidth=1.0)
        ax_price.set_title(f"{tf_labels[tf]} — price: actual vs. predicted")
        ax_price.set_xlabel("Actual price")
        ax_price.set_ylabel("Predicted price")
        ax_price.grid(True, alpha=0.25)

        ax_delta = axes[row, 1]
        ax_delta.scatter(r["true_delta"], r["pred_delta"], s=8, alpha=0.4, color="tab:orange")
        lo_d = float(min(r["true_delta"].min(), r["pred_delta"].min()))
        hi_d = float(max(r["true_delta"].max(), r["pred_delta"].max()))
        ax_delta.plot([lo_d, hi_d], [lo_d, hi_d], color="black", linestyle="--", linewidth=1.0)
        ax_delta.set_title(f"{tf_labels[tf]} — delta (bps): actual vs. predicted")
        ax_delta.set_xlabel("Actual delta (bps)")
        ax_delta.set_ylabel("Predicted delta (bps)")
        ax_delta.grid(True, alpha=0.25)

    fig.suptitle(title, fontsize=15, y=1.02)
    fig.tight_layout()
    plt.show()


plot_scatter(
    results,
    TF_LABELS,
    timeframes,
    f"HARN — {ASSET} — Seed {SEED} — Actual vs. Predicted Scatter",
)

In [ ]:
# ---------------------------------------------------------------------------
# Final experiment summary
# ---------------------------------------------------------------------------
print("=" * 60)
print("EXPERIMENT SUMMARY")
print("=" * 60)
print(f"Asset          : {ASSET}")
print(f"Seed           : {SEED}")
print(f"Checkpoint     : {checkpoint_path.relative_to(PROJECT_ROOT)}")
print(f"Dataset        : {test_payload_path.relative_to(PROJECT_ROOT)}")
print(f"Timeframes     : {timeframes}")
print(f"Test samples   : {n_predictions}")
print("-" * 60)
for tf in timeframes:
    da = directional_accuracy(results[tf]["true_delta"], results[tf]["pred_delta"])
    rmse = regression_metrics(results[tf]["true_price"], results[tf]["pred_price"])["RMSE"]
    print(f"{TF_LABELS[tf]:>6s} : Directional Accuracy = {da:6.2f}%   Price RMSE = {rmse:.6f}")
print("=" * 60)
print()
print("To evaluate a different experiment, change ASSET / SEED in the configuration")
print("cell above and Run All. No other code changes are required.")